# RuREBus: канонический preprocessing

Ноутбук запускает `src/preprocessing.py` и создаёт модель-независимый обработанный слой:

```text
rurebus_data/
├── RuREBus/        # raw: не изменяется
└── processed/
    ├── train/
    ├── validation/
    ├── test/
    ├── manifest.csv
    ├── duplicates.csv
    └── preprocessing_report.json
```

BIO, токенизация RuBERT, окна и `NO_RELATION` здесь намеренно не создаются.

In [ ]:
# 1. Подключение Google Drive и конфигурация
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import runpy
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_DIR = Path("/content/drive/MyDrive/NER_RuREBus_project")
DATA_ROOT = PROJECT_DIR / "rurebus_data"
RAW_ROOT = DATA_ROOT / "RuREBus"
PROCESSED_ROOT = DATA_ROOT / "processed"
PYPROJECT_PATH = PROJECT_DIR / "pyproject.toml"
BOOTSTRAP = PROJECT_DIR / "colab_bootstrap.py"

VALIDATION_SIZE = 0.20
RANDOM_SEED = 42
SEARCH_TRIALS = 5000
OVERWRITE = True

for required_path in (RAW_ROOT, PYPROJECT_PATH, BOOTSTRAP):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print(f"Raw:       {RAW_ROOT}")
print(f"Processed: {PROCESSED_ROOT}")
print(f"Project:   {PYPROJECT_PATH}")

In [ ]:
# 2. Установка пакета и запуск preprocessing
bootstrap_project = runpy.run_path(str(BOOTSTRAP))["bootstrap_project"]
bootstrap_project(PROJECT_DIR)

command = [
    sys.executable,
    "-m", "rurebus_ie.data.preprocessing",
    "--data-root", str(DATA_ROOT),
    "--validation-size", str(VALIDATION_SIZE),
    "--seed", str(RANDOM_SEED),
    "--search-trials", str(SEARCH_TRIALS),
]
if OVERWRITE:
    command.append("--overwrite")

print("Команда:", " ".join(command))
subprocess.run(command, check=True)
print("Preprocessing завершён.")

In [ ]:
# 3. Загрузка manifest и отчётов
manifest = pd.read_csv(PROCESSED_ROOT / "manifest.csv")
duplicates = pd.read_csv(PROCESSED_ROOT / "duplicates.csv")
report = json.loads((PROCESSED_ROOT / "preprocessing_report.json").read_text(encoding="utf-8"))

display(pd.DataFrame(report["split"]).T)
print("Проверки:")
display(pd.Series(report["checks"], name="result").to_frame())
display(manifest.head())

In [ ]:
# 4. Независимая проверка отсутствия утечек и наличия файлов
train = manifest[manifest["split"] == "train"]
validation = manifest[manifest["split"] == "validation"]
test = manifest[manifest["split"] == "test"]

assert set(train["source_id"]).isdisjoint(set(validation["source_id"]))
assert set(train["text_sha256"]).isdisjoint(set(validation["text_sha256"]))
assert set(train["text_sha256"]).isdisjoint(set(test["text_sha256"]))
assert set(validation["text_sha256"]).isdisjoint(set(test["text_sha256"]))
assert manifest["text_sha256"].nunique() == len(manifest)

missing_files = []
for row in manifest.itertuples(index=False):
    for column in ("processed_txt_path", "processed_ann_path"):
        path = DATA_ROOT / getattr(row, column)
        if not path.is_file():
            missing_files.append(str(path))
assert not missing_files, missing_files[:10]

print("Source leakage: 0")
print("Text-hash leakage: 0")
print(f"Проверено файловых пар: {len(manifest)}")

In [ ]:
# 5. Распределение сущностей и отношений по split
def expand_json_counts(frame, column):
    rows = []
    for item in frame.itertuples(index=False):
        for label, count in json.loads(getattr(item, column)).items():
            rows.append({"split": item.split, "label": label, "count": count})
    if not rows:
        return pd.DataFrame(columns=["split", "label", "count"])
    return pd.DataFrame(rows).groupby(["split", "label"], as_index=False)["count"].sum()

entity_distribution = expand_json_counts(manifest, "entity_types")
relation_distribution = expand_json_counts(manifest, "relation_types")

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
sns.barplot(data=entity_distribution, x="label", y="count", hue="split", ax=axes[0])
axes[0].set(title="Сущности по split", xlabel="Тип", ylabel="Количество")
sns.barplot(data=relation_distribution, x="label", y="count", hue="split", ax=axes[1])
axes[1].set(title="Отношения по split", xlabel="Тип", ylabel="Количество")
plt.tight_layout()
plt.show()

display(pd.crosstab(manifest["split"], columns="documents"))

In [ ]:
# 6. Отчёт о дублях
print(f"Групп дублей: {duplicates['text_sha256'].nunique()}")
print(f"Отброшено копий: {(~duplicates['selected'].astype(bool)).sum()}")
print(
    "Групп с различиями ANN:",
    duplicates.loc[~duplicates["annotation_equals_selected"].astype(bool), "text_sha256"].nunique(),
)
display(duplicates.sort_values(["text_sha256", "selected"], ascending=[True, False]))

In [ ]:
# 7. Итоговые пути для следующих ноутбуков
for split in ("train", "validation", "test"):
    split_dir = PROCESSED_ROOT / split
    print(f"{split:10s}: {split_dir} | TXT={len(list(split_dir.glob('*.txt')))}, ANN={len(list(split_dir.glob('*.ann')))}")

print(f"Manifest: {PROCESSED_ROOT / 'manifest.csv'}")
print("Обработанный BRAT-корпус готов к следующему этапу: tokenization + BIO.")